In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pyspark.sql.functions import col, isnan, count, when, explode, split, desc

# Giả định df_ratings, df_movies, df_tags đã được load từ HDFS như bước trước

print("=== 1. ĐÁNH GIÁ TÍNH TOÀN VẸN DỮ LIỆU ===")
# Đếm số lượng Null trong Ratings
null_counts = df_ratings.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df_ratings.columns])
print("Số lượng Null trong tập Ratings:")
null_counts.show()

# Kiểm tra tính nhất quán (Có movieId nào trong ratings không tồn tại trong movies?)
orphans = df_ratings.join(df_movies, on="movieId", how="left_anti")
print(f"Số lượng tương tác bị lỗi khóa ngoại (không tìm thấy phim): {orphans.count()}")


print("\n=== 2. PHÂN TÍCH ĐỘ THƯA THỚT (SPARSITY) ===")
num_ratings = df_ratings.count()
num_users = df_ratings.select("userId").distinct().count()
num_movies = df_movies.select("movieId").count()
sparsity = 1.0 - (num_ratings / (num_users * num_movies))
print(f"Tổng số User: {num_users}")
print(f"Tổng số Movie: {num_movies}")
print(f"Tổng số Rating: {num_ratings}")
print(f"Độ thưa thớt của ma trận: {sparsity * 100:.4f}%")


print("\n=== 3. PHÂN PHỐI ĐIỂM ĐÁNH GIÁ (RATING DISTRIBUTION) ===")
# Dùng Spark đếm số lượng, sau đó đưa về Pandas để vẽ đồ thị
rating_dist_df = df_ratings.groupBy("rating").count().orderBy("rating").toPandas()

plt.figure(figsize=(10, 5))
sns.barplot(x="rating", y="count", data=rating_dist_df, color="steelblue")
plt.title("Phân phối các mức điểm đánh giá (Rating Distribution)", fontsize=14)
plt.xlabel("Mức điểm (Stars)", fontsize=12)
plt.ylabel("Số lượng đánh giá", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show() # Lưu ảnh này để chèn vào Word báo cáo


print("\n=== 4. PHÂN TÍCH THỂ LOẠI (GENRES DISTRIBUTION) ===")
# Tách chuỗi 'Action|Adventure' thành nhiều dòng và đếm
genres_df = df_movies.withColumn("genre", explode(split(col("genres"), "\|")))
top_genres = genres_df.groupBy("genre").count().orderBy(desc("count")).toPandas()

plt.figure(figsize=(12, 6))
sns.barplot(y="genre", x="count", data=top_genres, color="seagreen")
plt.title("Số lượng phim theo từng thể loại (Genres Distribution)", fontsize=14)
plt.xlabel("Số lượng phim", fontsize=12)
plt.ylabel("Thể loại", fontsize=12)
plt.show() # Lưu ảnh này để chèn vào Word báo cáo


print("\n=== 5. PHÂN TÍCH TOP TAGS TỪ NGƯỜI DÙNG ===")
# df_tags là dataframe đọc từ tags.csv
df_tags = spark.read.csv(HDFS_PATH + "tags.csv", header=True, inferSchema=True)
# Đếm tần suất các tag được gõ nhiều nhất (chuyển về chữ thường để gom nhóm chính xác)
from pyspark.sql.functions import lower
top_tags_df = df_tags.withColumn("tag_lower", lower(col("tag"))) \
                     .groupBy("tag_lower").count().orderBy(desc("count")).limit(15).toPandas()

plt.figure(figsize=(10, 6))
sns.barplot(y="tag_lower", x="count", data=top_tags_df, color="coral")
plt.title("Top 15 Tags được người dùng sử dụng nhiều nhất", fontsize=14)
plt.xlabel("Số lượt gắn thẻ", fontsize=12)
plt.ylabel("Từ khóa (Tag)", fontsize=12)
plt.show()

ModuleNotFoundError: No module named 'seaborn'